# Sleep Staging on the Sleep Physionet Dataset

**Classifying 30-second EEG epochs into 5 sleep stages (W, N1, N2, N3, R) using a CNN.**

| | |
|---|---|
| **Architecture** | [SleepStagerChambon2018](src/models/sleep_stager_chambon_2018.py) — a lightweight spatial-temporal ConvNet for raw multi-channel EEG ([Chambon et al., 2018](https://doi.org/10.1109/TNSRE.2018.2813138)), modified to use LeakyReLU |
| **Dataset** | [PhysioNet Sleep EDF](https://physionet.org/content/sleep-edfx/) |
| **Pipeline** | Raw EDF → Band-pass filter (0.5–30 Hz) → ICA artifact removal → 30s epoch extraction → Subject-wise split → Train CNN |
| **Config** | All hyperparameters in [`config.yaml`](config.yaml) |
| **Background** | See [`README.md`](README.md) for sleep staging theory, deep learning concepts, and setup instructions |

## Libraries and Config


In [ ]:
# This tells Jupyter to automatically detect changes in your .py files and reload them every time you execute a cell.
%load_ext autoreload
%autoreload 2

# this ensures that plots open in a new window
%matplotlib qt

In [ ]:
import mne
import torch
import numpy as np
import matplotlib
import pathlib
from tqdm.contrib.concurrent import thread_map, process_map
from functools import partial
import gc
import yaml

matplotlib.use("QtAgg") # use the Qt backend for interactive plotting
mne.set_log_level("ERROR")  # suppress MNE info and warnings for cleaner output

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print("Using GPU for computations. Training should be faster.")
else:
    print(
        "No GPU found. Using CPU for computations, training might be slower."
        "\n\nIf running on Google Colab, make sure to enable GPU acceleration in the notebook settings."
    )

In [ ]:
# load all the 
with open('config.yaml', 'r') as file:
    config = yaml.safe_load(file)

## Loading the data

We use MNE's built-in `fetch_data` to download the [PhysioNet Sleep EDF Expanded](https://physionet.org/content/sleep-edfx/) dataset. This downloads pairs of files for each recording:
- **PSG file** (`.edf`) — the raw polysomnography recording (EEG, EOG, and other channels)
- **Hypnogram file** — expert-annotated sleep stage labels for each 30-second window

Our `load_sleep_physionet_raw_data` function then:
1. Loads only the EEG and EOG channels (discards EMG, respiration, temperature, event markers)
2. Attaches the sleep stage annotations to the raw data
3. Crops excessive wake periods at the start/end of each recording (keeps only 30 min of wake before/after sleep), since long wake segments add noise without useful training signal
4. Renames channels to a cleaner format (e.g., `EEG Fpz-Cz` → `Fpz-Cz`)


In [ ]:
from mne.datasets.sleep_physionet.age import fetch_data

# fetch the data
sleep_physionet_data_fnames = fetch_data(
    subjects=range(83), # fetch data for all 83 subjects,
    recording=[1, 2], # fetch both recordings for each subject,,
    path=pathlib.Path(config["data"]["dir"]["s_p_data"]),
    on_missing="warn"
)

len(sleep_physionet_data_fnames)

In [ ]:
from src.utils import load_sleep_physionet_raw_data

# load all raw data using multi-threading
raws = thread_map(
    lambda x : load_sleep_physionet_raw_data(raw_fname=x[0], annot_fname=x[1]),
    sleep_physionet_data_fnames,
    max_workers=4,
    desc="Loading the Raw data (preload=False)"
)

In [ ]:
# sanity check: plot the first raw data to verify it loaded correctly
raws[0].plot()

## Preprocessing

### 1. Performing Low-Pass filtering

During the **Awake** stage, our brain primarily produces **Beta waves** (13-30 Hz). As we drift into deeper sleep, these frequencies slow down, eventually reaching **Delta waves** (0.5-4 Hz) in N3/N4 deep sleep. Even during REM sleep, the brain typically produces **Theta waves** (4-8 Hz). This means that for sleep staging, the most relevant EEG information lies below 30 Hz.

Therefore, we will apply a simple **low-pass filter** with a cutoff frequency of 30 Hz. This removes higher-frequency noise (like muscle artifacts or line noise) that could negatively impact our model's performance.

In [ ]:
from src.utils import filter_and_save_raw_data

# path where the filtered EEG data will be saved. 
s_p_data_filtered_dir = pathlib.Path(config["data"]["dir"]["s_p_data_filtered"])
s_p_data_filtered_dir.mkdir(exist_ok=True)

# create a partial function with fixed arguments
func = partial(
    filter_and_save_raw_data, 
    l_freq=None, 
    h_freq=30, 
    output_dir=s_p_data_filtered_dir
)

# Process EEG data in parallel across multiple CPU cores.
# Note: process_map creates isolated child processes. 
# 1. The original 'raws' list in this script remains UNCHANGED (unfiltered).
# 2. Each worker loads, filters, and saves its own copy to the disk.
# 3. This prevents RAM exhaustion by only processing 'max_workers' files at a time.
filtered_raw_files = process_map(
    func, 
    raws, 
    max_workers=4,
    chunksize=1, # Processes one file at a time per worker for stability
    desc="Filtering and Saving the Raw data"
) 

In [ ]:
# load the filtered raw data using multi-threading
filtered_raws = thread_map(
    mne.io.read_raw_fif,
    filtered_raw_files,
    max_workers=2,
    desc="Loading the filtered raw data (preload=False)"
)

As a sanity check, let's plot the Power Spectral Density (PSD) of one of the raw recordings.

> ### What is PSD?
> In simple terms, PSD is a static overview that shows the "strength" or "power" of each frequency in the entire signal. Since we applied a 30 Hz low-pass filter, we expect to see a significant drop in power for frequencies above 30 Hz in the plot below.

In [ ]:
filtered_raws[0].plot_psd()

### 2. Cleaning the data using ICA

**Independent Component Analysis (ICA)** is a technique that decomposes the mixed EEG signal into statistically independent source components. The key idea: the signal recorded at each electrode is a *mixture* of brain activity + artifacts (eye blinks, eye movements, muscle noise). ICA "unmixes" these sources so we can identify and remove the artifact components while keeping the brain signal intact.

Our `perform_ica` function:
1. **Fits ICA** on a wider-band filtered copy (1–40 Hz) of the raw data - this gives the algorithm more frequency information to separate sources accurately
2. **Detects EOG artifact components** - automatically finds ICA components that correlate with the EOG (eye movement) channel using `find_bads_eog`
3. **Removes** those artifact components from the original 0.5–30 Hz filtered data
4. **Drops the EOG channel** after cleaning - it was only needed as a reference for artifact detection; the model trains on EEG channels only

> **Why not just filter?** Filtering removes entire frequency bands, but eye blink artifacts overlap with the Delta (0.5–4 Hz) band that is critical for detecting deep sleep (N3). ICA surgically removes the artifact *source* without destroying the underlying Delta activity.

In [ ]:
from src.utils import perform_ica

# path where the filtered EEG data will be saved. 
sleep_physionet_data_ica_dir = pathlib.Path(config["data"]["dir"]["s_p_data_ica"])
sleep_physionet_data_ica_dir.mkdir(exist_ok=True)

partial_func = partial(
    perform_ica,
    output_dir=sleep_physionet_data_ica_dir
)

ica_cleaned_filtered_raw_files = process_map(
    partial_func,
    list(zip(raws, filtered_raws)),
    max_workers=4,
    chunksize=1,
    desc="Applying ICA"
)

# clear the memory 
del raws, filtered_raws
gc.collect()     

In [ ]:
# Loading ICA cleaned Raw data
sleep_physionet_data_ica_dir = pathlib.Path(config["data"]["dir"]["s_p_data_ica"]) # path where the filtered EEG data iss saved

ica_cleaned_filtered_raw_files = []
# get all the ica cleaned file names 
for s_p_data_ica_fname in sleep_physionet_data_ica_dir.iterdir():
    ica_cleaned_filtered_raw_files.append(pathlib.Path(s_p_data_ica_fname))

# load the data
ica_cleaned_filtered_raws = thread_map(
    mne.io.read_raw_fif,
    ica_cleaned_filtered_raw_files,
    max_workers=2,
    desc="Loading the ICA cleaned filtered raw data (preload=False)"
)

# clear the memory 
del ica_cleaned_filtered_raw_files
gc.collect()

### 3. Creating Epochs Dataset

Sleep staging labels are assigned per **epoch** - a fixed-length, non-overlapping time window. The AASM standard defines this as **30 seconds**. Here we:

1. **Extract epochs** - slice each continuous recording into consecutive 30-second windows, each labelled with the corresponding sleep stage (W, N1, N2, N3, R). The old R&K stages 3 and 4 are merged into a single N3 class to match the modern AASM standard.
2. **Per-epoch standardisation** - each epoch is z-scored independently (mean=0, std=1 per channel). This is preferred over global scaling because EEG amplitudes vary across subjects and even within a single night (due to electrode impedance drift, sweat, etc.). Local scaling forces the model to learn from *wave morphology* rather than absolute voltage.
3. **Wrap as PyTorch Dataset** - each epoch becomes a `(X, y)` sample where `X` has shape `(1, 2, 3000)` and `y` is an integer label `{0..4}`.

All recordings are processed in parallel and then concatenated into a single `ConcatDataset`.


In [ ]:
from torch.utils.data import ConcatDataset

from src.utils import create_epochs_ds


# define the partial function with fixed arguments for creating the epochs ds
partial_fn = partial(
    create_epochs_ds,
    eeg_epoch_duration=config["data"]["eeg"]["epochs_duration"]
)

# store epochs dataset in parallel across multiple CPU cores.
datasets = process_map(
    partial_fn,
    ica_cleaned_filtered_raws,
    max_workers=4,
    chunksize=1,
    desc="Processing and creating epochs datasets"
)

# concatinate all the datasets into one big "datasets"
dataset = ConcatDataset(datasets)
print(f"Total number of epochs in the combined dataset: {len(dataset)}")

## Making train, valid and test splits
To strictly avoid data leakage, we perform a subject-wise split or subject-aware splitting. Sleep patterns are highly individualistic; if a model sees Subject A's Recording 1 in the training set, it will perform artificially well on Subject A's Recording 2 in the test set. By ensuring all recordings from a specific subject are confined to strictly one split (Train, Val, or Test), we measure the model's ability to generalize to new, unseen people, which is the gold standard for clinical applications.

In [ ]:
from src.datasets import split_by_subject

# calculate number of subjects for splits (e.g., 20% Test, 20% Val)
total_subjects = len(
    np.unique(
        [ds.subject_id for ds in dataset.datasets]
    )
)
n_test_subjects = max(1, int(total_subjects * config["data"]["ds"]["test_subjects_pct"]))
n_val_subjects = max(1, int(total_subjects * config["data"]["ds"]["val_subjects_pct"]))

# perform split
train_ds, valid_ds, test_ds = split_by_subject(
    dataset=dataset,
    n_test_subjects=n_test_subjects,
    n_val_subjects=n_val_subjects
)

# Note: the epochs here means different chucks of that particular dataset / iteration
print(f"Train size: {len(train_ds)} epochs")
print(f"Val size: {len(valid_ds)} epochs")
print(f"Test size: {len(test_ds)} epochs")

### Class Imbalance:
Sleep stages are not equal. We spend much more time in N2 than N1. To prevent the model from just guessing "N2" all the time, we compute Class Weights. These weights will be passed to our Loss Function to make the model "pay more attention" to rare classes like N1 and REM.

In [ ]:
import pandas as pd


classes_mapping  = {
    0: "Sleep stage W", 
    1: "Sleep stage 1", 
    2: "Sleep stage 2", 
    3: "Sleep stage 3", 
    4: "Sleep stage R"}

# get all the labels from the training set
train_y = np.concatenate(
    [ds.epochs_labels for ds in train_ds.datasets]
)

# plot the distribution
ax = pd.Series(train_y).map(classes_mapping).value_counts().plot(kind="barh")
ax.set_xlabel("Number of training example")
ax.set_ylabel("Sleep Stage")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# calculate weights
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_y),
    y=train_y
)

# convert to a PyTorch tensor for the Loss Function later
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

for i, class_weight in enumerate(class_weights):
    print(f"{classes_mapping[i]} : {class_weight}")

## Creating the Neural Network (Architecture)

In this section, we define our Convolutional Neural Network (ConvNet) architecture.

We will use the architecture proposed by **Chambon et al. (2018)**. This specific network is highly regarded because it is specifically designed for multivariate EEG time-series data. It is lightweight, fast to train, and interpretable.

Before looking at the PyTorch code, let's break downn the intuition behind this architecture and the specific techniques it uses.

<div style="text-align: center;"> 
<img src="imgs/chambon_convnet.png" alt="Chambon 2018 Architecture Diagram" width="800"> <br /> Source: <a href="https://github.com/hubertjb/dl-eeg-tutorial/blob/main/sleep_staging_physionet.ipynb">Adapted from Banville et al. 2020 (Tutorial on Deep Learning on Sleep Data)</a> 
</div>

### The Input and Output
+ **Input:** A 30-second window (epoch) of EEG data. For our Physionet dataset, we use $C=2$ channels (sampled at 100 Hz), resulting in an input shape of `(1, 2, 3000)` - where `1` is a dummy "depth" channel required by PyTorch Conv2D layers, `2` is the number of EEF channels, and `3000` is the number of time samples.
+ **Output:** A 5-dimensional vector containing the probability for each of the 5 sleep stages - W, N1, N2, N3, R.

### The Chambon ConvNet Layers
The Chambon architecture is clever because it splits the feature extraction into two distinct steps: **Spatial** (across different electrodes on the head) and **Temporal** (across time).

#### 1. Spatial Convolution (The "Where")
+ **What it does:** The very first layer is a `Spatial Conv (C, 1)`. Instead of lookin at a chunk of time, this filter looks at one single time point across all $C$ channels simultaneously. For our 100 Hz data, one sample represents a tiny slice of time exactly 10 milliseconds long ($1s / 100$). This filter combines the voltage readings from all electrodes at that exact 10 ms instant before moving to the next sample.

+ **The Intuition:** Imagine you have electrodes at the front (Fpz) and back (Pz) of the head. Sometimes, a sleep event is best detected by looking at the *difference* or *combination* of signals from these two locations at the exact same millisecond. This layer acts like a "virtual electrode", learning the optimal linear combination of the physical chaneels to highlight sleep-relevant brain activity. 


#### 2. Permute (Reshaping)
+ **What is does:** This is a simple matrix transpose operation. It swaps the dimensions of the data tensor so that the newly created "virtual channels" from the spatial convolution are ready to be analysed over time.


#### 3. Temporal Convolutions (The "When")
+ **What it does:** Next are the `Temporal Conv (1, 50)` layers. These filters look at a single virtual channel but scan across a window of time (e.g., 50 time sample, which is 0.5 seconds at 100 Hz).

+ **The Intuition:** This is where the network looks for specific wave shapes or transient events. Fo example, it might learn a filter that perfectly matches the shape of a **Sleep Spindle (11-16 Hz oscillation)** or a **K-complex (sharp slow waves)**. Because of *translation invairance* (i.e., the property of a system, model, or function that produces the same output regardless of a shift in the position of the input data), it can find tese events no matter where they occur in the 30-second window.

#### 4. Non-Linearity: ReLU
+ **What it does:** After the temporal convolutions, the data passes through a ReLU (Rectified Linear Unit) activation function: $f(x) = \max(0, x)$.

> **Note:** Our implementation uses **LeakyReLU** instead of standard ReLU. LeakyReLU allows a small negative slope ($f(x) = x$ if $x > 0$, else $f(x) = 0.01x$), which prevents the "dying ReLU" problem - where neurons that output 0 for all inputs can never recover because their gradient is always 0.


+ **The Intuition:** Convolutions are purely linear math (multiplication and addition). If a network only had linear layers, no matter how deep it was, it would just behave like one giant linear regression model. ReLU introduces non-linearity, allowing the network to learn complex, non-linear boundaries between sleep staes, e.g., if the spindle power is above the thresold,  trigger strongly; otherwise, output zero
<div style="text-align: center;"> 
<img src="imgs/relu.png" alt="ReLU activation function" width="300"> <br /> Source: <a href="https://www.researchgate.net/figure/Graphic-representation-of-the-ReLU-activation-function_fig3_348703101">researchgate</a> 
</div>

#### 4. Max Pooling (Downsampling)
+ **What it does:** `Max pool (1, 13)` slides a window (e.g., 13 samples wide) across the time axis and only keeps the maximum value in that window.

+ **The Intuition:** Once the temporal convolution has detected a feature (like a K-complex), we don't necessarily care exactly which millisecond it happened at; we just care that it did happen in that gerenal timeframe. Max pooling achieves three things:
    1. *Reduces dimensionality* - making the network faster and lighter.
    2. *Provides local translation invariance* - a slight shift in the input doesn't change the pooled output.
    3. *Summarise features* - over longer time scales.
    
#### 6. Flatten & Dropout
+ **Flatten:** Takes the 3D feature maps and unrolls them into a single 1D vector so that can be fed into a standard classifier.

+ **Dropout:** Randomly "turns off" a percentage of neurons during training. This forces the network to not rely too heavily  on any single features (e.g., it can't just memorise one specific channel's noise),acting gas a powerful regularisation technique to prevent overfitting.

#### 7. Fully Connected (Dense) Layer
+ **What it does:** The final step. Every neuron from the flattened feature vector connects to 5 output neurons (our 5 sleep stages).

+ **The Intuition:** This later acts as the final judge. It looks at all the complex spatial-temporal features extracted by the previous layers (e.g., "I see hugh Delta power and no eye movement and a K-complex") and weighs them together to make a final decision: "This is 95% likely to be Stage N2 sleep."

In [ ]:
from src.models import SleepStagerChambon2018

# sampling rate
sfreq = ica_cleaned_filtered_raws[0].info['sfreq']
# number of EEG channels
n_channels = len(ica_cleaned_filtered_raws[0].ch_names)


# Intialise the model
model = SleepStagerChambon2018(
    eeg_epoch_duration=config["data"]["eeg"]["epochs_duration"],
    n_channels=n_channels,
    sfreq=sfreq,
    n_classes=5,
    n_spatial_filters=config["model"]["architecture"]["n_spatial_filters"],
    n_temporal_filters_l1=config["model"]["architecture"]["n_temporal_filters_l1"],
    n_temporal_filters_l2=config["model"]["architecture"]["n_temporal_filters_l2"],
    temp_conv_size_sec=config["model"]["architecture"]["temp_conv_size_sec"],
    max_pool_size_sec=config["model"]["architecture"]["max_pool_size_sec"],
    dropout_rate=config["model"]["architecture"]["dropout_rate"]
)

# move model to GPU if CUDA is avilable for significantly faster training
print(f"Using device: {device}")
model = model.to(device)

## Training, Evaluating and Testing the Model

This section wires together the model, data, optimiser, and loss function into a complete training pipeline. The workflow:

1. **DataLoaders** - wrap the datasets into batched iterators for efficient GPU feeding
2. **Optimiser + Loss + Scheduler** - define *how* the model learns (Adam), *what* it optimises (weighted cross-entropy), and *when* to adjust the learning rate (ReduceLROnPlateau)
3. **Training loop** - trains the model, validates each epoch, applies early stopping, and logs all metrics to MLflow


### 1. Define DataLoader objects

A `DataLoader` wraps a Dataset and provides batched iteration: it groups individual epochs into mini-batches, optionally shuffles them (for training), and handles multi-worker data loading. The training loader shuffles to prevent the model from memorising the order of examples. Validation and test loaders don't shuffle - order doesn't matter during evaluation, and deterministic order ensures reproducible results.


In [ ]:
from torch.utils.data import DataLoader

# Create DataLoaders
loader_train = DataLoader(
    train_ds,
    batch_size=config["data"]["ds"]["train_batch_size"],
    shuffle=True
)
loader_val = DataLoader(
    valid_ds,
    batch_size=config["data"]["ds"]["val_batch_size"],
    shuffle=False
)
loader_test = DataLoader(
    test_ds,
    batch_size=config["data"]["ds"]["val_batch_size"],
    shuffle=False
)

### 2. Setup Optimiser, Loss Function and LR Scheduler

Three components control how the model learns:

- **Loss function** (`CrossEntropyLoss`) - measures how wrong the model's predictions are. We pass in our computed `class_weights` so the loss penalises errors on rare stages (N1, REM) more heavily than common stages (N2).
- **Optimiser** (`Adam`) - determines how weights are updated using the gradients. Adam adapts the learning rate per-parameter based on recent gradient history, which typically converges faster than basic SGD.
- **LR Scheduler** (`ReduceLROnPlateau`) - monitors validation loss and reduces the learning rate when it stops improving, allowing the model to fine-tune in later epochs rather than overshooting the optimum.

> When setting up the optimiser we can also pass the `weight_decay=1e-4` param, which is a regularization technique used in neural network optimization to prevent models from overfitting. When calulating the new weights it simply adds a separate step that subtracts a tiny fraction of the weight's current value.
>
>```math
>w_{new} = w_{old} − ( learning\_rate × gradient ) − (weight\_decay × w_{old})
>```


In [ ]:
from torch.nn import CrossEntropyLoss
from torch.optim import Adam

# loss function
criterion = CrossEntropyLoss(
    weight=class_weights_tensor # we pass our previously calculated class_weights to handle the class imbalance
).to(device=device) # put the criterion to GPU/CPU

# optimiser
optimiser = Adam(
    params=model.parameters(),
    lr=float(config["model"]["optimiser"]["lr"]),
    weight_decay=float(config["model"]["optimiser"]["weight_decay"])
)

In [ ]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

# LR Schedular
schedular = ReduceLROnPlateau(
    optimizer=optimiser,
    mode=config["model"]["schedular"]["mode"], 
    factor=config["model"]["schedular"]["factor"], 
    patience=config["model"]["schedular"]["patience"],
    threshold=float(config["model"]["schedular"]["threshold"])
)

### 3. Training, Validating and Testing the model

 Before running the run below, you first have to start the mlflow local server via your terminal:
```bash
# Activate the .venv environment
source .venv/bin/activate

# Start the mlflow local server at port 5000
mlflow server -p 5000
```

After running the above commands successfully, the mlflow dashbord should be hosted at `http://localhost:5000/`

In [ ]:
import mlflow
from sklearn.metrics import cohen_kappa_score, balanced_accuracy_score

from src.utils import fit_model
from src.utils import evaluate


# point MLflow to mlflow local server and set an experiment name to group your runs
mlflow.set_tracking_uri(config["mlflow"]["tracking_uri"])
mlflow.set_experiment(experiment_name=config["mlflow"]["experiment_name"])
with mlflow.start_run(run_name=config["mlflow"]["run_name"], nested=True) as run:
    mlflow.log_params(config)

    print(f"Started MLflow Run ID: {run.info.run_id}")
    baseline_training_mlflow_run_id = run.info.run_id
    
    # fit the model - training and evaluating the model
    best_model, history = fit_model(
        model=model,
        loader_train=loader_train,
        loader_valid=loader_val,
        optimiser=optimiser,
        criterion=criterion,
        device=device,
        metric_fns={"cohen_kappa_score" : cohen_kappa_score}, # Using Cohen's Kappa to account for class imbalance
        n_epochs=config["model"]["training"]["n_epochs"],
        patience=config["model"]["training"]["early_stopping_patience"],
        schedular=schedular,
        max_grad_norm=config["model"]["training"]["max_grad_norm"],
        mlflow_run=True 
    )

    # log the final trained PyTorch model artifact to MLflow!
    # mlflow.pytorch.log_model(
    #     pytorch_model=best_model, 
    #     name="best_model_artifact"
    # )

    # --- Final Evaluation on the Test Set
    # get the Cohen Kappa Score
    test_loss, test_performance = evaluate(
        model=model,
        loader=loader_test,
        criterion=criterion,
        device=device,
        metric_fns={"cohen_kappa_score" : cohen_kappa_score, "balanced_accuracy_score" : balanced_accuracy_score}
    )

    mlflow.log_metrics({
        "test_loss" : test_loss,
        "test_cohen_kappa_score" : test_performance["cohen_kappa_score"],
        "test_balanced_accuracy_score" : test_performance["balanced_accuracy_score"]
    })

    print(f"\nFinal Test Set Performance:")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Cohen's kappa score: {test_performance["cohen_kappa_score"]:.4f}")
    print(f"Test Balanced accuracy score: {test_performance["balanced_accuracy_score"]:.4f}")